# PART D: System design brief

### Time spent: 20 mins


---
### LAYER 1: Store raw data (backlog of all versions of data from past scripts dated and stored)

- Python scripts / scrapers / data extractors scheduled to run at regular intervals using Airflow
- Raw responses tagged by date/time (jsons/csvs/etc) are stored in cloud object storage like GCS

#### Notes:
- Run scripts for data copllection as infrequently as possible to lower compute costs and lower risk of hitting api limits. Run monthly/weekly/daily/hourly depending on demand for the type of data down the line
- Ingest new data tagged by date/time to preserve backlog of data pulled

---
### LAYER 2: Databases (latest versions of raw data, easily accessible)

- Use BigQuery or equivalent (ideally same provider of raw data storage for compatability - in this case Google)
- Loading from raw storage to BigQuery to keep tables updates with latest versions of scrapers / ingested data (older versions still in raw storage)

#### Notes:
- Airflow to schedule loading from raw storage to database
- Add a table to database to monitor latest pulls so scripts can track what has and hasn't been pulled yet and use this logic to not excessively update database unnecessarily. Include tags like ingestion status, feature table freshness, missing values, anomalies, updated daily/hourly. Ingestion failures can be noted and flagged here.

---

### LAYER 3: Feature Generation (refined tables for visualisation/analysis/modelling)

- Compute features from raw/structured data stored in databases, and these new tables are also stored in the same database /cloud platform such as BigQuery
- Features can be generated in BigQuery using SQL or in Python after querying raw tables

#### Notes:

- Use Airflow tasks to orchestrate these scripts
- Only recompute features when tables are updated, always queries table in database built to monitor latest pulls before deciding if script needs to be run again

---

### LAYER 4: Consuption (live dashboards, staged modelling jobs etc.)

- Scripts for visualisation in BI tool directly such as Looker
- Scripts for pulling in data for modelling directly in Vertex AI for ML pipelines

#### Notes:

- Uses the nicely curated tables from Layer 3
- Dashboards can be scheduled automatically when these tables update